# 04 — Xuất model YOLO (.pt) sang ONNX (.onnx)

**Mục tiêu:** chuyển `best.pt` (bất kỳ notebook train nào — độ chín hay bệnh lá) sang `.onnx` để chạy trong trình duyệt với `ai/scripts/leaf_disease_tester.html` (hoặc công cụ tương tự cho nhánh độ chín sau này). Trình duyệt không chạy được `.pt` (PyTorch) trực tiếp, phải qua ONNX.

Notebook này **dùng chung cho cả 2 nhánh** — không hardcode class cụ thể, chỉ xuất định dạng.

## Trước khi chạy
1. **Add Input** → Kaggle Dataset chứa file `.pt` cần xuất (vd output đã Save Version của `03b_train_leaf_augmented.ipynb` hoặc `02c_train_ripeness_augmented.ipynb`).
2. Không cần GPU. Internet: **ON** (cần cài `onnxslim` để tối ưu file ONNX).
3. Nếu Kaggle Dataset chứa nhiều file `.pt` (vd cả `best.pt` lẫn `last.pt`), notebook sẽ liệt kê hết — xem kỹ Bước 1 trước khi để nó tự chọn, hoặc tự đặt `PT_OVERRIDE`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics", "onnx", "onnxslim"], check=False)

### Bước 1 — Liệt kê toàn bộ file `.pt` tìm thấy trong `/kaggle/input`
Không đoán file nào là đúng — in hết ra kèm kích thước + đường dẫn đầy đủ để tự xác nhận trước khi export.

In [ ]:
from pathlib import Path

all_pt_files = sorted(Path("/kaggle/input").rglob("*.pt"))

if not all_pt_files:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy file .pt nào trong /kaggle/input. Kiểm tra đã Add Input đúng Kaggle Dataset "
        "chứa model đã train (đã Save Version) chưa."
    )

print(f"Tìm thấy {len(all_pt_files)} file .pt:\n")
for p in all_pt_files:
    size_mb = p.stat().st_size / 1e6
    print(f"  [{size_mb:6.1f} MB] {p}")

# Đặt thủ công nếu muốn chọn khác với lựa chọn tự động bên dưới, vd:
# PT_OVERRIDE = Path("/kaggle/input/notebooks/.../models/tomato_leaf_disease_yolov8n_augmented.pt")
PT_OVERRIDE = None

### Bước 2 — Chọn file để export
Ưu tiên: `PT_OVERRIDE` (nếu đặt) → file nằm trong thư mục `models/` (bản sao đặt tên rõ ràng, do các notebook train tự lưu) → file tên chứa "best" (không lấy "last.pt", vốn là checkpoint cuối cùng chứ không phải tốt nhất) → nếu vẫn không rõ, lấy file đầu tiên và cảnh báo kiểm tra lại.

In [ ]:
def pick_pt_file(files):
    in_models_dir = [p for p in files if p.parent.name == "models"]
    if in_models_dir:
        return in_models_dir[0], "nam trong thu muc models/"
    best_named = [p for p in files if "best" in p.stem.lower()]
    if best_named:
        return best_named[0], "ten file chua 'best'"
    return files[0], "khong ro tieu chi, lay file dau tien -> KIEM TRA LAI cho chac"


if PT_OVERRIDE is not None:
    PT_PATH = PT_OVERRIDE
    reason = "PT_OVERRIDE do nguoi dung dat"
else:
    PT_PATH, reason = pick_pt_file(all_pt_files)

print(f"Đã chọn: {PT_PATH}")
print(f"Lý do: {reason}")
print(f"Kích thước: {PT_PATH.stat().st_size / 1e6:.1f} MB")

### Bước 3 — Export sang ONNX

In [ ]:
import shutil

from ultralytics import YOLO

# /kaggle/input chỉ đọc — model.export() lại mặc định lưu file .onnx ngay cạnh file .pt nguồn,
# nên phải copy .pt sang /kaggle/working (ghi được) trước rồi mới load + export từ đó.
WORKING_PT = Path("/kaggle/working") / PT_PATH.name
shutil.copy2(PT_PATH, WORKING_PT)

model = YOLO(str(WORKING_PT))
print("Class trong model:", model.names)

onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
onnx_path = Path(onnx_path)
print(f"\nĐã xuất: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

### Bước 4 — Kiểm tra file ONNX sau khi export
Kiểm tra model có thể được load bằng thư viện `onnx`, lấy kích thước input/output và kiểm tra ONNX checker.

In [ ]:
import onnx

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("✓ ONNX checker: PASS")
print("Input:")
for inp in onnx_model.graph.input:
    dims = []
    for d in inp.type.tensor_type.shape.dim:
        dims.append(d.dim_value if d.dim_value > 0 else d.dim_param)
    print(f"  - {inp.name}: {dims}")

print("Output:")
for out in onnx_model.graph.output:
    dims = []
    for d in out.type.tensor_type.shape.dim:
        dims.append(d.dim_value if d.dim_value > 0 else d.dim_param)
    print(f"  - {out.name}: {dims}")

### Bước 5 — Đóng gói artifact để tải xuống
File ONNX được copy sang `/kaggle/working/exports/`. Khi notebook chạy xong và **Save Version**, thư mục này nằm trong Output của Kaggle.

In [ ]:
EXPORT_DIR = Path("/kaggle/working/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

final_onnx_path = EXPORT_DIR / onnx_path.name
shutil.copy2(onnx_path, final_onnx_path)

print("=" * 60)
print("EXPORT HOÀN TẤT")
print(f"PT nguồn      : {PT_PATH}")
print(f"ONNX tạm      : {onnx_path}")
print(f"ONNX cuối     : {final_onnx_path}")
print(f"Kích thước    : {final_onnx_path.stat().st_size / 1e6:.2f} MB")
print("=" * 60)
print("Trong Kaggle: mở tab Output/Files và tải file ONNX trong thư mục exports/")

### Lưu ý khi dùng trong trình duyệt
- File `.pt` **không** chạy trực tiếp trong browser; sử dụng file `.onnx` vừa export.
- Với pipeline YOLO hiện tại, `imgsz=640` được giữ đúng như Bước 3.
- `model.names` được in ra để xác nhận class của model trước khi đưa sang frontend.
- Nếu Dataset có nhiều `.pt`, nên đặt `PT_OVERRIDE` thủ công để tránh export nhầm model.